In [8]:
# ============================================================
# 08_results_tables_figures.ipynb
# Main-paper results only
#
# Purpose:
#   1) Collect key outputs from existing folders
#   2) Build main comparison tables
#   3) Build H1 structural comparison tables
#   4) Build clean placebo summary tables
#   5) Copy main-text figures into one folder with clean names
#   6) Generate paper-ready summary plots
#
# This file does NOT re-estimate models.
# It only reads results already produced by earlier files.
# ============================================================

from pathlib import Path
from persim import wasserstein
import shutil
import warnings
import re

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
ROOT = Path(".").resolve()

DIRS = {
    "author": ROOT / "outputs_authorweight",
    "author_tda": ROOT / "outputs_authorweight_tda",
    "author_placebo": ROOT / "outputs_authorweight_placebo",
    "baseline": ROOT / "outputs_baseline",
    "baseline_tda": ROOT / "outputs_baseline_tda",
    "baseline_placebo": ROOT / "outputs_baseline_placebo",
    "topology": ROOT / "outputs_topology_augmented",
    "topology_check": ROOT / "outputs_topology_augmented_tda_check",
    "topology_placebo": ROOT / "outputs_topology_augmented_placebo",
}

OUT_DIR = ROOT / "outputs_results_tables_figures"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
SUPPORT_DIR = OUT_DIR / "support"
MANIFEST_DIR = OUT_DIR / "manifest"

for d in [OUT_DIR, TABLE_DIR, FIG_DIR, SUPPORT_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [p for p in DIRS.values() if p.exists()]

print("Output directory:", OUT_DIR)
print("Search roots:")
for p in SEARCH_ROOTS:
    print(" -", p)

# ------------------------------------------------------------
# 0.5) Clean stale files
# ------------------------------------------------------------
STALE_FILES = [
    FIG_DIR / "fig9_baseline_vs_topology_augmented_paths_and_centered_gaps.pdf",
    FIG_DIR / "fig14_author_vs_topology_paths_and_centered_gaps.pdf",
]

for p in STALE_FILES:
    if p.exists() and p.is_file():
        p.unlink()
        print(f"Removed stale file: {p}")

# ------------------------------------------------------------
# 1) Helpers
# ------------------------------------------------------------
def unique_paths(paths):
    seen = set()
    out = []
    for p in paths:
        rp = Path(p).resolve()
        if rp not in seen:
            seen.add(rp)
            out.append(rp)
    return out


def find_all(patterns, roots=None):
    if roots is None:
        roots = SEARCH_ROOTS
    if isinstance(patterns, str):
        patterns = [patterns]

    hits = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            hits.extend(root.rglob(pat))

    return unique_paths(hits)


def find_first(patterns, roots=None):
    hits = find_all(patterns, roots=roots)
    return hits[0] if hits else None


def find_first_prefer(patterns, preferred_roots=None):
    preferred_roots = preferred_roots or []
    hit = find_first(patterns, roots=preferred_roots)
    if hit is not None:
        return hit
    return find_first(patterns, roots=SEARCH_ROOTS)


def read_csv_safe(path):
    if path is None or not Path(path).exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        warnings.warn(f"Could not read csv: {path}\n{e}")
        return None


def copy_file(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def save_table(df, stem):
    csv_path = TABLE_DIR / f"{stem}.csv"
    txt_path = TABLE_DIR / f"{stem}.txt"

    df.to_csv(csv_path, index=False)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(df.to_string(index=False))

    return csv_path, txt_path


def find_col(df, include=None, exclude=None, exact=None):
    include = include or []
    exclude = exclude or []
    cols = list(df.columns)

    if exact is not None:
        for c in cols:
            if c.lower() == exact.lower():
                return c

    for c in cols:
        cl = c.lower()
        if all(s.lower() in cl for s in include) and not any(s.lower() in cl for s in exclude):
            return c

    return None


def find_first_existing_col(df, candidates):
    cols = list(df.columns)

    for c in candidates:
        if c in cols:
            return c

    lower_map = {c.lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    return None


def find_col_by_keywords(df, include_all=None, include_any=None, exclude_any=None):
    include_all = include_all or []
    include_any = include_any or []
    exclude_any = exclude_any or []

    for c in df.columns:
        cl = c.lower()

        if include_all and not all(s.lower() in cl for s in include_all):
            continue

        if include_any and not any(s.lower() in cl for s in include_any):
            continue

        if exclude_any and any(s.lower() in cl for s in exclude_any):
            continue

        return c

    return None


def nice_method_name(x):
    x = str(x).strip().lower()

    if x in {"author", "authorweight", "author_weight"}:
        return "author"

    if x == "baseline":
        return "baseline"

    if x in {"topology_augmented", "topology-augmented", "topology augmented", "topo"}:
        return "topology_augmented"

    return x


def extract_embedding_from_text(x):
    if pd.isna(x):
        return None

    s = str(x)
    m = re.search(r"(\d+)\s*[,._-]\s*(\d+)", s)

    if m:
        return f"({m.group(1)},{m.group(2)})"

    return s


def load_h1_diagram_csv(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing H1 diagram csv: {path}")

    df = pd.read_csv(path)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(num_cols) < 2:
        raise ValueError(f"{path} does not have at least two numeric columns.")

    out = df[num_cols[:2]].copy()
    out.columns = ["birth", "death"]
    out = out.replace([np.inf, -np.inf], np.nan).dropna()

    if len(out) == 0:
        return np.empty((0, 2))

    return out[["birth", "death"]].to_numpy(dtype=float)


def safe_wasserstein(dgm_a, dgm_b):
    dgm_a = np.asarray(dgm_a, dtype=float).reshape(-1, 2)
    dgm_b = np.asarray(dgm_b, dtype=float).reshape(-1, 2)

    if len(dgm_a) == 0 and len(dgm_b) == 0:
        return 0.0

    return float(wasserstein(dgm_a, dgm_b))


def find_file_in_dir(base_dir, patterns):
    base_dir = Path(base_dir)

    if isinstance(patterns, str):
        patterns = [patterns]

    if not base_dir.exists():
        return None

    hits = []
    for pat in patterns:
        hits.extend(base_dir.rglob(pat))

    hits = unique_paths(hits)
    return hits[0] if hits else None


def centered_gap_from_series(treated, synthetic, dates, treatment_start):
    treated = pd.Series(treated).astype(float)
    synthetic = pd.Series(synthetic).astype(float)
    dates = pd.to_datetime(dates)

    gap = treated - synthetic
    pre_mean = gap.loc[dates < treatment_start].mean()

    return gap - pre_mean


def load_author_topology_path_comparison(author_path, topology_path, treatment_start):
    """
    Build Figure 14 data:
      - Missouri
      - author-weight synthetic
      - topology-augmented synthetic
      - author gap
      - topology gap
      - author centered gap
      - topology centered gap
    """
    author_df = read_csv_safe(author_path)
    topology_df = read_csv_safe(topology_path)

    if author_df is None or author_df.empty:
        raise ValueError(f"Cannot read authorweight_df: {author_path}")

    if topology_df is None or topology_df.empty:
        raise ValueError(f"Cannot read topology_augmented_df: {topology_path}")

    author_df = author_df.copy()
    topology_df = topology_df.copy()

    author_date_col = find_first_existing_col(author_df, ["date"])
    topo_date_col = find_first_existing_col(topology_df, ["date"])

    if author_date_col is None:
        raise ValueError("Cannot find date column in authorweight_df.")

    if topo_date_col is None:
        raise ValueError("Cannot find date column in topology_augmented_df.")

    author_df[author_date_col] = pd.to_datetime(author_df[author_date_col])
    topology_df[topo_date_col] = pd.to_datetime(topology_df[topo_date_col])

    author_treated_col = find_first_existing_col(
        author_df,
        ["treated", "missouri", "treated_level", "treated_author"]
    )

    topo_treated_col = find_first_existing_col(
        topology_df,
        ["treated", "missouri", "treated_level", "treated_topology_augmented"]
    )

    if author_treated_col is None:
        author_treated_col = find_col_by_keywords(
            author_df,
            include_any=["treated", "missouri"],
            exclude_any=["gap", "synthetic", "centered"]
        )

    if topo_treated_col is None:
        topo_treated_col = find_col_by_keywords(
            topology_df,
            include_any=["treated", "missouri"],
            exclude_any=["gap", "synthetic", "centered"]
        )

    if author_treated_col is None:
        raise ValueError("Cannot find treated/Missouri column in authorweight_df.")

    if topo_treated_col is None:
        raise ValueError("Cannot find treated/Missouri column in topology_augmented_df.")

    author_synth_col = find_first_existing_col(
        author_df,
        [
            "synthetic_author",
            "synthetic_authorweight",
            "synthetic_author_weight",
            "author_synthetic",
            "synthetic"
        ]
    )

    if author_synth_col is None:
        author_synth_col = find_col_by_keywords(
            author_df,
            include_all=["synthetic"],
            include_any=["author"],
            exclude_any=["gap", "centered"]
        )

    if author_synth_col is None:
        raise ValueError("Cannot find author synthetic column.")

    topo_synth_col = find_first_existing_col(
        topology_df,
        [
            "synthetic_topology_augmented",
            "synthetic_topology",
            "synthetic_topo",
            "topology_augmented_synthetic",
            "topology_augmented"
        ]
    )

    if topo_synth_col is None:
        topo_synth_col = find_col_by_keywords(
            topology_df,
            include_any=["topology", "topo", "synthetic"],
            exclude_any=["gap", "centered", "rmspe", "lambda"]
        )

    if topo_synth_col is None:
        raise ValueError("Cannot find topology-augmented synthetic column.")

    author_keep = author_df[[author_date_col, author_treated_col, author_synth_col]].copy()
    author_keep.columns = ["date", "missouri_author_source", "synthetic_author"]

    topo_keep = topology_df[[topo_date_col, topo_treated_col, topo_synth_col]].copy()
    topo_keep.columns = ["date", "missouri_topology_source", "synthetic_topology_augmented"]

    out = (
        author_keep
        .merge(topo_keep, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    if out.empty:
        raise ValueError("Merged author/topology path dataframe is empty.")

    out["missouri"] = out["missouri_author_source"]

    diff = np.nanmax(np.abs(out["missouri_author_source"] - out["missouri_topology_source"]))
    if np.isfinite(diff) and diff > 1e-8:
        warnings.warn(
            f"Missouri treated series differs across author/topology files. "
            f"Max abs diff={diff:.8f}. Using author source as Missouri."
        )

    out["gap_author"] = out["missouri"] - out["synthetic_author"]
    out["gap_topology_augmented"] = out["missouri"] - out["synthetic_topology_augmented"]

    out["centered_gap_author"] = centered_gap_from_series(
        out["missouri"],
        out["synthetic_author"],
        out["date"],
        treatment_start
    )

    out["centered_gap_topology_augmented"] = centered_gap_from_series(
        out["missouri"],
        out["synthetic_topology_augmented"],
        out["date"],
        treatment_start
    )

    keep_cols = [
        "date",
        "missouri",
        "synthetic_author",
        "synthetic_topology_augmented",
        "gap_author",
        "gap_topology_augmented",
        "centered_gap_author",
        "centered_gap_topology_augmented"
    ]

    return out[keep_cols]


# ------------------------------------------------------------
# 2) Discover key files
# ------------------------------------------------------------
KEY_FILES = {}

KEY_FILES["author_diagnostics"] = find_first_prefer(
    ["authorweight_diagnostics.csv"],
    preferred_roots=[DIRS["author"]]
)

KEY_FILES["baseline_diagnostics"] = find_first_prefer(
    ["baseline_diagnostics.csv"],
    preferred_roots=[DIRS["baseline"]]
)

KEY_FILES["author_tda_summary"] = find_first_prefer(
    ["authorweight_embedding_summary.csv"],
    preferred_roots=[DIRS["author_tda"]]
)

KEY_FILES["baseline_tda_summary"] = find_first_prefer(
    ["baseline_embedding_summary.csv"],
    preferred_roots=[DIRS["baseline_tda"]]
)

KEY_FILES["baseline_placebo_summary"] = find_first_prefer(
    ["baseline_placebo_summary.csv"],
    preferred_roots=[DIRS["baseline_placebo"]]
)

KEY_FILES["author_placebo_summary"] = find_first_prefer(
    [
        "*author*placebo*summary*.csv",
        "*authorweight*placebo*summary*.csv"
    ],
    preferred_roots=[DIRS["author_placebo"]]
)

KEY_FILES["topology_pre_post_comparison"] = find_first_prefer(
    [
        "baseline_vs_topology_augmented_prepost_metrics.csv",
        "baseline_vs_topology_augmented_metrics.csv",
        "*baseline*topology*prepost*metrics*.csv",
        "*baseline*topology*metr*.csv"
    ],
    preferred_roots=[DIRS["topology"]]
)

KEY_FILES["authorweight_df"] = find_first_prefer(
    ["authorweight_df.csv"],
    preferred_roots=[DIRS["author"]]
)

KEY_FILES["topology_augmented_df"] = find_first_prefer(
    [
        "topology_augmented_df.csv",
        "*topology*augmented*df*.csv"
    ],
    preferred_roots=[DIRS["topology"]]
)

print("\nDiscovered key files:")
for k, v in KEY_FILES.items():
    print(f"{k:35s}: {v}")


# ------------------------------------------------------------
# 3) Build main pre/post comparison table
# ------------------------------------------------------------
TREATMENT_START = pd.Timestamp("2011-04-01")


def parse_diagnostics_table(path, method_name):
    df = read_csv_safe(path)

    if df is None or df.empty:
        return None

    row = df.iloc[0]

    # prefer explicit date columns
    treatment_col = find_first_existing_col(
        df,
        ["treatment_date", "treatment_start", "start_date"]
    )

    pre_col = find_col(df, include=["pre", "rmspe"])
    post_col = find_col(df, include=["post", "rmspe"], exclude=["ratio"])
    ratio_col = find_col(df, include=["ratio"]) or find_col(df, include=["post_pre"])
    n_pre_col = find_col(df, include=["n_periods_pre"]) or find_col(df, include=["n_pre"])
    n_post_col = find_col(df, include=["n_periods_post"]) or find_col(df, include=["n_post"])

    return {
        "method": method_name,
        "treatment_start": row[treatment_col] if treatment_col else np.nan,
        "n_pre": row[n_pre_col] if n_pre_col else np.nan,
        "n_post": row[n_post_col] if n_post_col else np.nan,
        "pre_rmspe": row[pre_col] if pre_col else np.nan,
        "post_rmspe": row[post_col] if post_col else np.nan,
        "post_pre_ratio": row[ratio_col] if ratio_col else np.nan,
        "source_file": str(path),
    }


pre_post_rows = []

author_diag_row = parse_diagnostics_table(KEY_FILES["author_diagnostics"], "author")
if author_diag_row:
    pre_post_rows.append(author_diag_row)

baseline_diag_row = parse_diagnostics_table(KEY_FILES["baseline_diagnostics"], "baseline")
if baseline_diag_row:
    pre_post_rows.append(baseline_diag_row)

topo_df = read_csv_safe(KEY_FILES["topology_pre_post_comparison"])

if topo_df is not None and not topo_df.empty:
    lower_cols = {c.lower(): c for c in topo_df.columns}

    needed = {"method", "pre_rmspe", "post_rmspe", "post_pre_ratio"}

    if needed.issubset(set(lower_cols.keys())):
        rename_map = {lower_cols[k]: k for k in needed}
        tmp = topo_df.rename(columns=rename_map).copy()
        tmp["method"] = tmp["method"].map(nice_method_name)
        tmp = tmp[tmp["method"].eq("topology_augmented")]

        keep = [
            c for c in
            ["method", "treatment_start", "n_pre", "n_post", "pre_rmspe", "post_rmspe", "post_pre_ratio"]
            if c in tmp.columns
        ]

        if len(tmp) > 0:
            row = tmp.iloc[0][keep].to_dict()
            row["source_file"] = str(KEY_FILES["topology_pre_post_comparison"])
            pre_post_rows.append(row)

pre_post_df = pd.DataFrame(pre_post_rows)

if not pre_post_df.empty:
    ordered_cols = [
        "method", "treatment_start", "n_pre", "n_post",
        "pre_rmspe", "post_rmspe", "post_pre_ratio", "source_file"
    ]

    pre_post_df = pre_post_df[[c for c in ordered_cols if c in pre_post_df.columns]]
    pre_post_df["method"] = pre_post_df["method"].map(nice_method_name)

    method_order = ["author", "baseline", "topology_augmented"]
    pre_post_df["method"] = pd.Categorical(pre_post_df["method"], categories=method_order, ordered=True)
    pre_post_df = pre_post_df.sort_values("method").reset_index(drop=True)

    save_table(pre_post_df, "main_pre_post_comparison")

    print("\nSaved: main_pre_post_comparison")
    print(pre_post_df.round(6).to_string(index=False))
else:
    print("\nWarning: no pre/post comparison table could be built.")


# ------------------------------------------------------------
# 4) Build H1 structural comparison tables
# ------------------------------------------------------------
def parse_embedding_summary(path, method_name):
    df = read_csv_safe(path)

    if df is None or df.empty:
        return None

    embed_col = find_col(df, include=["embedding"]) or find_col(df, include=["dim", "delay"])
    h1_wass_col = find_col(df, include=["h1", "wasserstein"])
    h1_bott_col = find_col(df, include=["h1", "bottleneck"])

    if embed_col is None or h1_wass_col is None:
        return None

    out = df.copy()
    out["embedding"] = out[embed_col].apply(extract_embedding_from_text)
    out["method"] = method_name
    out["h1_wasserstein"] = out[h1_wass_col]

    if h1_bott_col is not None:
        out["h1_bottleneck"] = out[h1_bott_col]

    keep = [c for c in ["method", "embedding", "h1_wasserstein", "h1_bottleneck"] if c in out.columns]
    out = out[keep].copy()
    out["source_file"] = str(path)

    return out


h1_frames = []

author_h1 = parse_embedding_summary(KEY_FILES["author_tda_summary"], "author")
if author_h1 is not None:
    h1_frames.append(author_h1)

baseline_h1 = parse_embedding_summary(KEY_FILES["baseline_tda_summary"], "baseline")
if baseline_h1 is not None:
    h1_frames.append(baseline_h1)

h1_main_df = pd.concat(h1_frames, ignore_index=True) if h1_frames else pd.DataFrame()

if not h1_main_df.empty:
    save_table(h1_main_df, "main_h1_author_baseline_comparison")
    print("\nSaved: main_h1_author_baseline_comparison")
    print(h1_main_df.round(6).to_string(index=False))
else:
    print("\nWarning: no author/baseline H1 comparison table could be built.")


# ------------------------------------------------------------
# 4.1) Build baseline vs topology H1 comparison table
# ------------------------------------------------------------
TOPO_CHECK_DIR = DIRS["topology_check"]


def find_topology_treated_h1(emb_tag):
    patterns = [
        f"treated_H1_diagram_{emb_tag}.csv",
        f"missouri_H1_diagram_{emb_tag}.csv",
        f"*treated*H1*diagram*{emb_tag}*.csv",
        f"*missouri*H1*diagram*{emb_tag}*.csv",
    ]
    return find_file_in_dir(TOPO_CHECK_DIR, patterns)


def find_baseline_h1(emb_tag):
    patterns = [
        f"baseline_H1_diagram_{emb_tag}.csv",
        f"*baseline*H1*diagram*{emb_tag}*.csv",
        f"*baseline*H1*{emb_tag}*.csv",
    ]
    return find_file_in_dir(TOPO_CHECK_DIR, patterns)


def find_topology_augmented_h1(emb_tag):
    patterns = [
        f"topology_augmented_H1_diagram_{emb_tag}.csv",
        f"*topology_augmented*H1*diagram*{emb_tag}*.csv",
        f"*topology*augmented*H1*diagram*{emb_tag}*.csv",
    ]
    return find_file_in_dir(TOPO_CHECK_DIR, patterns)


topo_h1_rows = []
missing_topo_h1_files = []

for emb_tag, emb_label in [("3_1", "(3,1)"), ("2_1", "(2,1)")]:
    treated_path = find_topology_treated_h1(emb_tag)
    baseline_path = find_baseline_h1(emb_tag)
    topology_path = find_topology_augmented_h1(emb_tag)

    if treated_path is None:
        missing_topo_h1_files.append(f"treated/Missouri H1 diagram for {emb_label}")
        continue

    if baseline_path is None:
        missing_topo_h1_files.append(f"baseline H1 diagram for {emb_label}")
        continue

    if topology_path is None:
        missing_topo_h1_files.append(f"topology-augmented H1 diagram for {emb_label}")
        continue

    treated_dgm = load_h1_diagram_csv(treated_path)
    baseline_dgm = load_h1_diagram_csv(baseline_path)
    topology_dgm = load_h1_diagram_csv(topology_path)

    h1_baseline = safe_wasserstein(treated_dgm, baseline_dgm)
    h1_topology = safe_wasserstein(treated_dgm, topology_dgm)

    topo_h1_rows.append({
        "embedding": emb_label,
        "h1_wasserstein_baseline": h1_baseline,
        "h1_wasserstein_topology_augmented": h1_topology,
        "h1_absolute_reduction": h1_baseline - h1_topology,
        "h1_percent_reduction": (
            100 * (h1_baseline - h1_topology) / h1_baseline
            if h1_baseline != 0 else np.nan
        ),
        "treated_h1_source_file": str(treated_path),
        "baseline_h1_source_file": str(baseline_path),
        "topology_augmented_h1_source_file": str(topology_path),
    })

if missing_topo_h1_files:
    print("\nWarning: missing files for baseline-vs-topology H1 comparison:")
    for item in missing_topo_h1_files:
        print(" -", item)

topo_h1_clean = pd.DataFrame(topo_h1_rows)

if not topo_h1_clean.empty:
    topo_h1_clean = topo_h1_clean.sort_values("embedding").reset_index(drop=True)

    save_table(topo_h1_clean, "main_h1_baseline_vs_topology_comparison")

    print("\nSaved: main_h1_baseline_vs_topology_comparison")
    print(topo_h1_clean.round(6).to_string(index=False))
else:
    print("\nWarning: no valid baseline-vs-topology H1 comparison table could be built.")


# ------------------------------------------------------------
# 4.5) Build author vs topology H1 comparison table
# ------------------------------------------------------------
AUTHOR_TDA_DIR = DIRS["author_tda"]


def find_author_treated_h1(emb_tag):
    patterns = [
        f"authorweight_treated_H1_diagram_{emb_tag}.csv",
        f"*authorweight*treated*H1*diagram*{emb_tag}*.csv",
        f"*treated*H1*diagram*{emb_tag}*.csv",
        f"*missouri*H1*diagram*{emb_tag}*.csv",
    ]
    return find_file_in_dir(AUTHOR_TDA_DIR, patterns)


def find_author_synthetic_h1(emb_tag):
    patterns = [
        f"authorweight_synthetic_H1_diagram_{emb_tag}.csv",
        f"authorweight_synthetic_author_H1_diagram_{emb_tag}.csv",
        f"*authorweight*synthetic*H1*diagram*{emb_tag}*.csv",
        f"*author*synthetic*H1*diagram*{emb_tag}*.csv",
        f"*synthetic*author*H1*diagram*{emb_tag}*.csv",
        f"*synthetic*H1*diagram*{emb_tag}*.csv",
    ]
    return find_file_in_dir(AUTHOR_TDA_DIR, patterns)


author_vs_topology_rows = []
missing_author_topology_files = []

for emb_tag, emb_label in [("3_1", "(3,1)"), ("2_1", "(2,1)")]:
    author_treated_path = find_author_treated_h1(emb_tag)
    author_synth_path = find_author_synthetic_h1(emb_tag)

    topo_treated_path = find_topology_treated_h1(emb_tag)
    topo_synth_path = find_topology_augmented_h1(emb_tag)

    h1_author = np.nan

    if author_treated_path is not None and author_synth_path is not None:
        author_treated_dgm = load_h1_diagram_csv(author_treated_path)
        author_synth_dgm = load_h1_diagram_csv(author_synth_path)
        h1_author = safe_wasserstein(author_treated_dgm, author_synth_dgm)
        author_source_mode = "diagram_csv"
    else:
        # Fallback: use authorweight_embedding_summary.csv if synthetic diagram is missing.
        if h1_main_df is not None and not h1_main_df.empty:
            temp_author = h1_main_df[
                (h1_main_df["method"].eq("author")) &
                (h1_main_df["embedding"].eq(emb_label))
            ]

            if len(temp_author) > 0:
                h1_author = float(temp_author.iloc[0]["h1_wasserstein"])
                author_source_mode = "embedding_summary"
            else:
                author_source_mode = "missing"
        else:
            author_source_mode = "missing"

    if np.isnan(h1_author):
        missing_author_topology_files.append(f"author H1 distance for {emb_label}")
        continue

    if topo_treated_path is None:
        missing_author_topology_files.append(f"topology treated/Missouri H1 diagram for {emb_label}")
        continue

    if topo_synth_path is None:
        missing_author_topology_files.append(f"topology-augmented H1 diagram for {emb_label}")
        continue

    topo_treated_dgm = load_h1_diagram_csv(topo_treated_path)
    topo_synth_dgm = load_h1_diagram_csv(topo_synth_path)
    h1_topology = safe_wasserstein(topo_treated_dgm, topo_synth_dgm)

    author_vs_topology_rows.append({
        "embedding": emb_label,
        "h1_wasserstein_author": h1_author,
        "h1_wasserstein_topology_augmented": h1_topology,
        "h1_absolute_reduction_vs_author": h1_author - h1_topology,
        "h1_percent_reduction_vs_author": (
            100 * (h1_author - h1_topology) / h1_author
            if h1_author != 0 else np.nan
        ),
        "author_source_mode": author_source_mode,
        "author_treated_h1_source_file": str(author_treated_path) if author_treated_path else "",
        "author_synthetic_h1_source_file": str(author_synth_path) if author_synth_path else "",
        "topology_treated_h1_source_file": str(topo_treated_path),
        "topology_augmented_h1_source_file": str(topo_synth_path),
    })

if missing_author_topology_files:
    print("\nWarning: missing files or values for author-vs-topology H1 comparison:")
    for item in missing_author_topology_files:
        print(" -", item)

author_vs_topology_h1_df = pd.DataFrame(author_vs_topology_rows)

if not author_vs_topology_h1_df.empty:
    author_vs_topology_h1_df = (
        author_vs_topology_h1_df
        .sort_values("embedding")
        .reset_index(drop=True)
    )

    save_table(author_vs_topology_h1_df, "main_h1_author_vs_topology_comparison")

    print("\nSaved: main_h1_author_vs_topology_comparison")
    print(author_vs_topology_h1_df.round(6).to_string(index=False))
else:
    print("\nWarning: no author-vs-topology H1 comparison table could be built.")


# ------------------------------------------------------------
# 5) Build formal placebo summary table
# ------------------------------------------------------------
placebo_frames = []

baseline_placebo_df = read_csv_safe(KEY_FILES["baseline_placebo_summary"])
if baseline_placebo_df is not None and not baseline_placebo_df.empty:
    temp = baseline_placebo_df.copy()
    temp["method"] = "baseline"
    placebo_frames.append(temp)

author_placebo_df = read_csv_safe(KEY_FILES["author_placebo_summary"])
if author_placebo_df is not None and not author_placebo_df.empty:
    temp = author_placebo_df.copy()
    temp["method"] = "author"
    placebo_frames.append(temp)
else:
    print("\nNote: author placebo summary not found; main_formal_placebo_summary will include baseline only if available.")

combined_placebo_df = pd.concat(placebo_frames, ignore_index=True) if placebo_frames else pd.DataFrame()

if not combined_placebo_df.empty:
    save_table(combined_placebo_df, "main_formal_placebo_summary")
    print("\nSaved: main_formal_placebo_summary")
else:
    print("\nWarning: no combined placebo summary table could be built.")


# ------------------------------------------------------------
# 6) Copy main-text figures into one folder with clean names
# ------------------------------------------------------------
FIG_SPECS = [
    {
        "patterns": ["author_replication_figure13.pdf"],
        "target": "fig01_author_replication_figure13.pdf",
        "caption": "Author replication figure."
    },
    {
        "patterns": ["baseline_replication_figure13.pdf"],
        "target": "fig02_baseline_replication_figure13.pdf",
        "caption": "Baseline SCM replication figure."
    },
    {
        "patterns": ["baseline_vs_author_paths.pdf"],
        "target": "fig03_baseline_vs_author_paths.pdf",
        "caption": "Baseline versus author path comparison."
    },
    {
        "patterns": ["baseline_standardized_pre_series.pdf"],
        "target": "fig04_baseline_standardized_pre_series.pdf",
        "caption": "Baseline standardized pre-treatment series."
    },
    {
        "patterns": ["authorweight_standardized_pre_series.pdf"],
        "target": "fig05_authorweight_standardized_pre_series.pdf",
        "caption": "Author-weight standardized pre-treatment series."
    },
    {
        "patterns": ["baseline_placebo_3_1_histogram.pdf"],
        "target": "fig06_baseline_placebo_3_1_histogram.pdf",
        "caption": "Baseline placebo histogram for embedding (3,1)."
    },
    {
        "patterns": ["baseline_placebo_2_1_histogram.pdf"],
        "target": "fig07_baseline_placebo_2_1_histogram.pdf",
        "caption": "Baseline placebo histogram for embedding (2,1)."
    },
    {
        "patterns": ["pre_treatment_standardized_path_comparison.pdf"],
        "target": "fig08_pre_treatment_standardized_path_baseline_vs_topology.pdf",
        "caption": "Pre-treatment standardized paths: Missouri, baseline SCM, and topology-augmented SCM."
    },
    {
        "patterns": ["baseline_vs_topology_augmented_paths.pdf"],
        "target": "fig09_baseline_vs_topology_augmented_paths_and_centered_gaps.pdf",
        "caption": "Missouri and synthetic paths, plus centered gaps: baseline versus topology-augmented SCM."
    },
]

figure_manifest_rows = []

for spec in FIG_SPECS:
    src = find_first_prefer(spec["patterns"], preferred_roots=list(DIRS.values()))

    if src is not None and Path(src).exists():
        dst = FIG_DIR / spec["target"]
        copy_file(src, dst)

        figure_manifest_rows.append({
            "target_file": dst.name,
            "source_file": str(src),
            "caption": spec["caption"],
            "use": "main_text"
        })
    else:
        print(f"Warning: figure source not found for {spec['target']}")

figure_manifest_df = pd.DataFrame(figure_manifest_rows)

if not figure_manifest_df.empty:
    figure_manifest_df.to_csv(MANIFEST_DIR / "main_figure_manifest.csv", index=False)

print("\nCopied main figures:")
if not figure_manifest_df.empty:
    print(figure_manifest_df.to_string(index=False))
else:
    print("No figure files found.")


# ------------------------------------------------------------
# 7) Create paper-ready summary plots
# ------------------------------------------------------------
created_plot_rows = []

# ------------------------------------------------------------
# 7.1 Pre vs post RMSPE scatter
# ------------------------------------------------------------
if not pre_post_df.empty and {"method", "pre_rmspe", "post_rmspe"}.issubset(pre_post_df.columns):
    fig, ax = plt.subplots(figsize=(6.2, 4.8))

    ax.scatter(pre_post_df["pre_rmspe"], pre_post_df["post_rmspe"])

    for _, r in pre_post_df.iterrows():
        ax.annotate(
            str(r["method"]),
            (r["pre_rmspe"], r["post_rmspe"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9
        )

    ax.set_xlabel("Pre RMSPE")
    ax.set_ylabel("Post RMSPE")
    ax.set_title("Method comparison: pre vs post RMSPE")
    ax.grid(True, linewidth=0.5, alpha=0.7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    out_path = FIG_DIR / "fig10_method_pre_vs_post_rmspe.pdf"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

    created_plot_rows.append({
        "target_file": out_path.name,
        "source_file": "generated_in_08",
        "caption": "Scatter plot of pre RMSPE versus post RMSPE across methods.",
        "use": "main_text"
    })


# ------------------------------------------------------------
# 7.2 Post/pre ratio bar chart
# ------------------------------------------------------------
if not pre_post_df.empty and {"method", "post_pre_ratio"}.issubset(pre_post_df.columns):
    temp = pre_post_df.dropna(subset=["post_pre_ratio"]).copy()

    if len(temp) > 0:
        fig, ax = plt.subplots(figsize=(6.2, 4.8))

        ax.bar(temp["method"].astype(str), temp["post_pre_ratio"])

        ax.set_xlabel("Method")
        ax.set_ylabel("Post / pre RMSPE ratio")
        ax.set_title("Method comparison: post/pre RMSPE ratio")
        ax.grid(True, axis="y", linewidth=0.5, alpha=0.7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()

        out_path = FIG_DIR / "fig11_method_post_pre_ratio.pdf"
        plt.savefig(out_path, bbox_inches="tight")
        plt.close(fig)

        created_plot_rows.append({
            "target_file": out_path.name,
            "source_file": "generated_in_08",
            "caption": "Bar chart of post/pre RMSPE ratio across methods.",
            "use": "main_text"
        })


# ------------------------------------------------------------
# 7.3 H1 comparison plot: author vs baseline
# ------------------------------------------------------------
if not h1_main_df.empty and {"method", "embedding", "h1_wasserstein"}.issubset(h1_main_df.columns):
    temp = h1_main_df.copy()

    piv = temp.pivot_table(
        index="embedding",
        columns="method",
        values="h1_wasserstein",
        aggfunc="first"
    )

    if {"author", "baseline"}.issubset(piv.columns):
        piv = piv.dropna(subset=["author", "baseline"], how="any")

        if len(piv) > 0:
            fig, ax = plt.subplots(figsize=(6.5, 4.8))

            piv[["author", "baseline"]].plot(kind="bar", ax=ax)

            ax.set_xlabel("Embedding")
            ax.set_ylabel("H1 Wasserstein distance")
            ax.set_title("Author vs baseline H1 structural mismatch")
            ax.grid(True, axis="y", linewidth=0.5, alpha=0.7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            plt.tight_layout()

            out_path = FIG_DIR / "fig12_author_vs_baseline_h1_comparison.pdf"
            plt.savefig(out_path, bbox_inches="tight")
            plt.close(fig)

            created_plot_rows.append({
                "target_file": out_path.name,
                "source_file": "generated_in_08",
                "caption": "Bar chart of H1 Wasserstein mismatch for author and baseline methods across shared embeddings.",
                "use": "main_text"
            })


# ------------------------------------------------------------
# 7.4 H1 comparison plot: baseline vs topology
# ------------------------------------------------------------
if topo_h1_clean is not None and not topo_h1_clean.empty:
    required_cols = {
        "embedding",
        "h1_wasserstein_baseline",
        "h1_wasserstein_topology_augmented"
    }

    if required_cols.issubset(topo_h1_clean.columns):
        temp = topo_h1_clean.copy()

        fig, ax = plt.subplots(figsize=(6.5, 4.8))

        x = np.arange(len(temp))
        width = 0.35

        ax.bar(
            x - width / 2,
            temp["h1_wasserstein_baseline"],
            width=width,
            label="baseline"
        )

        ax.bar(
            x + width / 2,
            temp["h1_wasserstein_topology_augmented"],
            width=width,
            label="topology_augmented"
        )

        ax.set_xticks(x)
        ax.set_xticklabels(temp["embedding"])
        ax.set_xlabel("Embedding")
        ax.set_ylabel("H1 Wasserstein distance")
        ax.set_title("Baseline vs topology-augmented H1 mismatch")
        ax.legend()
        ax.grid(True, axis="y", linewidth=0.5, alpha=0.7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()

        out_path = FIG_DIR / "fig13_baseline_vs_topology_h1_comparison.pdf"
        plt.savefig(out_path, bbox_inches="tight")
        plt.close(fig)

        created_plot_rows.append({
            "target_file": out_path.name,
            "source_file": "generated_in_08",
            "caption": "Bar chart of H1 Wasserstein mismatch for baseline and topology-augmented methods.",
            "use": "main_text"
        })

        print("\nGenerated Figure 13:")
        print(" -", out_path.name)
    else:
        print("\nWarning: Figure 13 skipped because required columns are missing.")
else:
    print("\nWarning: Figure 13 skipped because topo_h1_clean is empty.")


# ------------------------------------------------------------
# 7.5 H1 comparison plot: author vs topology
# ------------------------------------------------------------
if author_vs_topology_h1_df is not None and not author_vs_topology_h1_df.empty:
    required_cols = {
        "embedding",
        "h1_wasserstein_author",
        "h1_wasserstein_topology_augmented"
    }

    if required_cols.issubset(author_vs_topology_h1_df.columns):
        temp = author_vs_topology_h1_df.copy()

        fig, ax = plt.subplots(figsize=(6.5, 4.8))

        x = np.arange(len(temp))
        width = 0.35

        ax.bar(
            x - width / 2,
            temp["h1_wasserstein_author"],
            width=width,
            label="author"
        )

        ax.bar(
            x + width / 2,
            temp["h1_wasserstein_topology_augmented"],
            width=width,
            label="topology_augmented"
        )

        ax.set_xticks(x)
        ax.set_xticklabels(temp["embedding"])
        ax.set_xlabel("Embedding")
        ax.set_ylabel("H1 Wasserstein distance")
        ax.set_title("Author vs topology-augmented H1 mismatch")
        ax.legend()
        ax.grid(True, axis="y", linewidth=0.5, alpha=0.7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()

        out_path = FIG_DIR / "fig13b_author_vs_topology_h1_comparison.pdf"
        plt.savefig(out_path, bbox_inches="tight")
        plt.close(fig)

        created_plot_rows.append({
            "target_file": out_path.name,
            "source_file": "generated_in_08",
            "caption": "Bar chart of H1 Wasserstein mismatch for author and topology-augmented methods.",
            "use": "main_text"
        })

        print("\nGenerated Figure 13b:")
        print(" -", out_path.name)


# ------------------------------------------------------------
# 7.6 Figure 14: author vs topology path and centered-gap comparison
# ------------------------------------------------------------
try:
    author_topology_path_df = load_author_topology_path_comparison(
        KEY_FILES["authorweight_df"],
        KEY_FILES["topology_augmented_df"],
        TREATMENT_START
    )

    save_table(
        author_topology_path_df,
        "main_author_vs_topology_paths_and_centered_gaps"
    )

    support_csv_path = SUPPORT_DIR / "fig14_author_vs_topology_paths_and_centered_gaps_data.csv"
    author_topology_path_df.to_csv(support_csv_path, index=False)

    fig, axes = plt.subplots(2,1,
        figsize=(8, 12),
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1]}
    )

    ax0, ax1 = axes

    xticks = pd.to_datetime([
        "2009-01-01",
        "2010-01-01",
        "2011-01-01",
        "2012-01-01",
        "2013-01-01"
    ])

    xlabels = [d.strftime("Jan%Y") for d in xticks]

    # Upper panel: paths
    ax0.plot(
        author_topology_path_df["date"],
        author_topology_path_df["missouri"],
        label="Missouri",
        linewidth=1.8
    )

    ax0.plot(
        author_topology_path_df["date"],
        author_topology_path_df["synthetic_author"],
        label="Synthetic Missouri: author weights",
        linewidth=1.6
    )

    ax0.plot(
        author_topology_path_df["date"],
        author_topology_path_df["synthetic_topology_augmented"],
        label="Synthetic Missouri: topology-augmented",
        linewidth=1.6
    )

    ax0.axvline(
        TREATMENT_START,
        linestyle="--",
        linewidth=1.2,
        label="Treatment start"
    )

    ax0.set_title("Missouri vs synthetic paths: Author Weights and Topology-Augmented SCM")
    ax0.set_ylabel("Deseasonalized unemployment rate")
    ax0.set_xticks(xticks)
    ax0.set_xticklabels(xlabels)
    ax0.tick_params(axis="x", labelbottom=True)
    ax0.grid(True, linewidth=0.5, alpha=0.7)
    ax0.legend()
    ax0.spines["top"].set_visible(False)
    ax0.spines["right"].set_visible(False)

    # Lower panel: centered gaps
    ax1.plot(
        author_topology_path_df["date"],
        author_topology_path_df["centered_gap_author"],
        label="Centered gap: author weights",
        linewidth=1.6
    )

    ax1.plot(
        author_topology_path_df["date"],
        author_topology_path_df["centered_gap_topology_augmented"],
        label="Centered gap: topology-augmented",
        linewidth=1.6
    )

    ax1.axhline(0, linewidth=1.0)

    ax1.axvline(
        TREATMENT_START,
        linestyle="--",
        linewidth=1.2,
        label="Treatment start"
    )

    ax1.set_title("Centered gaps: Author Weights vs Topology-Augmented SCM")
    ax1.set_ylabel("MO - synthetic control")
    ax1.set_xticks(xticks)
    ax1.set_xticklabels(xlabels)
    ax1.grid(True, linewidth=0.5, alpha=0.7)
    ax1.legend()
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    plt.tight_layout()

    out_path = FIG_DIR / "fig14_author_vs_topology_paths_and_centered_gaps.pdf"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

    created_plot_rows.append({
        "target_file": out_path.name,
        "source_file": "generated_in_08",
        "caption": "Missouri and synthetic paths, plus centered gaps: author weights versus topology-augmented SCM.",
        "use": "main_text"
    })

    print("\nGenerated Figure 14:")
    print(" -", out_path.name)
    print("Saved Figure 14 data:")
    print(" - main_author_vs_topology_paths_and_centered_gaps.csv")
    print(" - main_author_vs_topology_paths_and_centered_gaps.txt")
    print(" -", support_csv_path.name)

except Exception as e:
    print("\nWarning: Figure 14 author-vs-topology path comparison could not be generated.")
    print("Reason:", e)


# ------------------------------------------------------------
# 7.7 Save generated plot manifest
# ------------------------------------------------------------
if created_plot_rows:
    created_plot_df = pd.DataFrame(created_plot_rows)
    created_plot_df.to_csv(MANIFEST_DIR / "generated_main_plots_manifest.csv", index=False)

    print("\nGenerated main plots:")
    print(created_plot_df.to_string(index=False))
else:
    print("\nNo new summary plots were generated.")


# ------------------------------------------------------------
# 8) Build master manifest for file 08 outputs
# ------------------------------------------------------------
manifest_rows = []

for p in sorted(TABLE_DIR.glob("*")):
    if not p.is_file():
        continue

    manifest_rows.append({
        "section": "tables",
        "file_name": p.name,
        "file_type": p.suffix.lower().replace(".", ""),
        "folder": str(TABLE_DIR),
        "recommended_use": "main_text"
    })

for p in sorted(FIG_DIR.glob("*")):
    if not p.is_file():
        continue

    manifest_rows.append({
        "section": "figures",
        "file_name": p.name,
        "file_type": p.suffix.lower().replace(".", ""),
        "folder": str(FIG_DIR),
        "recommended_use": "main_text"
    })

for p in sorted(SUPPORT_DIR.glob("*")):
    if not p.is_file():
        continue

    manifest_rows.append({
        "section": "support",
        "file_name": p.name,
        "file_type": p.suffix.lower().replace(".", ""),
        "folder": str(SUPPORT_DIR),
        "recommended_use": "supporting_data"
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_DIR / "08_main_outputs_manifest.csv", index=False)

with open(MANIFEST_DIR / "08_main_outputs_manifest.txt", "w", encoding="utf-8") as f:
    if len(manifest_df) > 0:
        f.write(manifest_df.to_string(index=False))
    else:
        f.write("No outputs found.")

print("\nSaved manifest:")
print(MANIFEST_DIR / "08_main_outputs_manifest.csv")


# ------------------------------------------------------------
# 9) Final console summary
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("08_results_tables_figures finished")
print("=" * 72)

print("\nTables created:")
for p in sorted(TABLE_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name)

print("\nFigures collected/generated:")
for p in sorted(FIG_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name)

print("\nSupport files:")
for p in sorted(SUPPORT_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name)

print("\nManifest files:")
for p in sorted(MANIFEST_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name)

Output directory: /home/jupyter/topological fit check on scm/outputs_results_tables_figures
Search roots:
 - /home/jupyter/topological fit check on scm/outputs_authorweight
 - /home/jupyter/topological fit check on scm/outputs_authorweight_tda
 - /home/jupyter/topological fit check on scm/outputs_baseline
 - /home/jupyter/topological fit check on scm/outputs_baseline_tda
 - /home/jupyter/topological fit check on scm/outputs_baseline_placebo
 - /home/jupyter/topological fit check on scm/outputs_topology_augmented
 - /home/jupyter/topological fit check on scm/outputs_topology_augmented_tda_check
Removed stale file: /home/jupyter/topological fit check on scm/outputs_results_tables_figures/figures/fig14_author_vs_topology_paths_and_centered_gaps.pdf

Discovered key files:
author_diagnostics                 : /home/jupyter/topological fit check on scm/outputs_authorweight/authorweight_diagnostics.csv
baseline_diagnostics               : /home/jupyter/topological fit check on scm/outputs_bas